# Healthcare Claims Anomaly Detection
## Outlier Claims per Insurance Provider
**Topic:** Clinical Decision Making & Pattern Recognition in Healthcare  
**Technique:** Isolation Forest (Unsupervised Anomaly Detection)  
**TPO Focus:** Payment Integrity — detecting statistically anomalous claims per insurer


In [ ]:
# ── CELL 1: Install & Import ──────────────────────────────────────────────────
# Run this cell first. If kagglehub is not installed, it will install it.
import subprocess, sys

def install(pkg):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])

for pkg in ['kagglehub', 'pandas', 'numpy', 'scikit-learn', 'plotly', 'ipywidgets']:
    install(pkg)

print('✓ All packages ready')

In [ ]:
# ── CELL 2: Load Dataset ──────────────────────────────────────────────────────
import kagglehub
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Download dataset
path = kagglehub.dataset_download('prasad22/healthcare-dataset')
print(f'Dataset path: {path}')

import os
# Find the CSV file in the downloaded folder
csv_file = None
for f in os.listdir(path):
    if f.endswith('.csv'):
        csv_file = os.path.join(path, f)
        break

df = pd.read_csv(csv_file)

# Standardize column names
df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_')

print(f'\n✓ Dataset loaded: {df.shape[0]:,} rows × {df.shape[1]} columns')
print(f'\nColumns: {list(df.columns)}')
df.head(3)

In [ ]:
# ── CELL 3: Exploratory Data Analysis ────────────────────────────────────────
print('=' * 50)
print('DATASET OVERVIEW')
print('=' * 50)

print(f'\nShape          : {df.shape}')
print(f'Missing values : {df.isnull().sum().sum()}')

print('\n── Insurance Providers ──')
print(df['insurance_provider'].value_counts())

print('\n── Billing Amount Stats ──')
print(df['billing_amount'].describe().round(2))

print('\n── Medical Conditions ──')
print(df['medical_condition'].value_counts())

print('\n── Admission Types ──')
print(df['admission_type'].value_counts())

In [ ]:
# ── CELL 4: Feature Engineering ──────────────────────────────────────────────
from sklearn.preprocessing import LabelEncoder, StandardScaler

df_model = df.copy()

# Parse dates and derive length of stay
df_model['date_of_admission'] = pd.to_datetime(df_model['date_of_admission'])
df_model['discharge_date']    = pd.to_datetime(df_model['discharge_date'])
df_model['length_of_stay']    = (df_model['discharge_date'] - df_model['date_of_admission']).dt.days

# Encode categorical features
le = LabelEncoder()
categorical_cols = ['medical_condition', 'insurance_provider', 'admission_type', 'medication']
for col in categorical_cols:
    df_model[col + '_encoded'] = le.fit_transform(df_model[col].astype(str))

# Compute insurer-level statistics — this is the key feature
# Billing amount relative to what's EXPECTED for that insurer+condition combo
insurer_condition_stats = (
    df_model
    .groupby(['insurance_provider', 'medical_condition'])['billing_amount']
    .agg(mean_billing='mean', std_billing='std')
    .reset_index()
)

df_model = df_model.merge(insurer_condition_stats, on=['insurance_provider', 'medical_condition'], how='left')

# Z-score: how many std devs is THIS claim from the insurer+condition average?
df_model['billing_zscore'] = (
    (df_model['billing_amount'] - df_model['mean_billing']) /
    df_model['std_billing'].replace(0, 1)
)

# Billing amount relative to insurer average (ratio feature)
insurer_avg = df_model.groupby('insurance_provider')['billing_amount'].mean().rename('insurer_avg_billing')
df_model = df_model.join(insurer_avg, on='insurance_provider')
df_model['billing_vs_insurer_avg'] = df_model['billing_amount'] / df_model['insurer_avg_billing']

# Final feature set for the model
FEATURES = [
    'billing_amount',          # raw claim value
    'billing_zscore',          # deviation from insurer+condition norm
    'billing_vs_insurer_avg',  # ratio vs insurer average
    'length_of_stay',          # clinical context
    'medical_condition_encoded',
    'insurance_provider_encoded',
    'admission_type_encoded',
]

X = df_model[FEATURES].fillna(0)

# Scale features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print('✓ Features engineered')
print(f'  Model input shape : {X_scaled.shape}')
print(f'  Features used     : {FEATURES}')

In [ ]:
# ── CELL 5: Isolation Forest — Anomaly Detection ─────────────────────────────
from sklearn.ensemble import IsolationForest

# contamination = expected fraction of anomalies
# 0.05 means we expect ~5% of claims to be outliers — realistic for insurance fraud
iso_forest = IsolationForest(
    n_estimators=200,      # more trees = more stable scores
    contamination=0.05,    # ~5% anomaly rate
    random_state=42,
    n_jobs=-1              # use all CPU cores
)

# Fit and predict
# -1 = anomaly (outlier), +1 = normal
df_model['anomaly_label'] = iso_forest.fit_predict(X_scaled)

# Anomaly score: more negative = more anomalous
df_model['anomaly_score']  = iso_forest.score_samples(X_scaled)

# Normalize score to 0-100 for readability (100 = most anomalous)
s = df_model['anomaly_score']
df_model['anomaly_pct'] = ((s - s.max()) / (s.min() - s.max()) * 100).round(1)

# Flag column for easy filtering
df_model['is_anomaly'] = (df_model['anomaly_label'] == -1).astype(int)

n_anomalies = df_model['is_anomaly'].sum()
n_total     = len(df_model)

print('=' * 50)
print('ISOLATION FOREST RESULTS')
print('=' * 50)
print(f'Total claims    : {n_total:,}')
print(f'Flagged anomalies: {n_anomalies:,} ({n_anomalies/n_total*100:.1f}%)')
print(f'Normal claims   : {n_total - n_anomalies:,}')

print('\n── Anomalies per insurer ──')
anomaly_by_insurer = (
    df_model.groupby('insurance_provider')
    .agg(
        total_claims   = ('is_anomaly', 'count'),
        anomaly_count  = ('is_anomaly', 'sum'),
        avg_billing    = ('billing_amount', 'mean'),
        avg_anomaly_score = ('anomaly_pct', 'mean')
    )
    .assign(anomaly_rate=lambda x: (x['anomaly_count']/x['total_claims']*100).round(1))
    .sort_values('anomaly_rate', ascending=False)
    .round(2)
)
print(anomaly_by_insurer.to_string())

In [ ]:
# ── CELL 6: Visualizations ────────────────────────────────────────────────────
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

normal   = df_model[df_model['is_anomaly'] == 0]
anomalies = df_model[df_model['is_anomaly'] == 1]

# ── Plot 1: Scatter — Billing Amount vs Z-score, colored by anomaly ──────────
fig1 = go.Figure()

fig1.add_trace(go.Scatter(
    x=normal['billing_zscore'],
    y=normal['billing_amount'],
    mode='markers',
    name='Normal claim',
    marker=dict(color='steelblue', size=4, opacity=0.4),
    hovertemplate=(
        '<b>Normal Claim</b><br>'
        'Insurer: %{customdata[0]}<br>'
        'Condition: %{customdata[1]}<br>'
        'Billing: $%{y:,.0f}<br>'
        'Z-score: %{x:.2f}<extra></extra>'
    ),
    customdata=normal[['insurance_provider','medical_condition']].values
))

fig1.add_trace(go.Scatter(
    x=anomalies['billing_zscore'],
    y=anomalies['billing_amount'],
    mode='markers',
    name='Anomalous claim',
    marker=dict(color='crimson', size=7, opacity=0.8, symbol='x'),
    hovertemplate=(
        '<b>⚠ ANOMALOUS CLAIM</b><br>'
        'Insurer: %{customdata[0]}<br>'
        'Condition: %{customdata[1]}<br>'
        'Billing: $%{y:,.0f}<br>'
        'Z-score: %{x:.2f}<br>'
        'Anomaly score: %{customdata[2]:.1f}/100<extra></extra>'
    ),
    customdata=anomalies[['insurance_provider','medical_condition','anomaly_pct']].values
))

fig1.update_layout(
    title='Outlier Claims per Insurer — Billing Amount vs Z-score',
    xaxis_title='Billing Z-score (deviation from insurer+condition average)',
    yaxis_title='Billing Amount ($)',
    height=500,
    legend=dict(orientation='h', y=1.08)
)
fig1.show()

# ── Plot 2: Bar — Anomaly rate per insurer ────────────────────────────────────
fig2 = px.bar(
    anomaly_by_insurer.reset_index(),
    x='insurance_provider',
    y='anomaly_rate',
    color='anomaly_rate',
    color_continuous_scale='RdYlGn_r',
    text='anomaly_rate',
    title='Anomaly Rate (%) by Insurance Provider',
    labels={'anomaly_rate': 'Anomaly rate (%)', 'insurance_provider': 'Insurer'},
    height=420
)
fig2.update_traces(texttemplate='%{text}%', textposition='outside')
fig2.update_layout(coloraxis_showscale=False, xaxis_tickangle=-20)
fig2.show()

# ── Plot 3: Box — Billing distribution normal vs anomaly per insurer ──────────
fig3 = px.box(
    df_model,
    x='insurance_provider',
    y='billing_amount',
    color='is_anomaly',
    color_discrete_map={0: 'steelblue', 1: 'crimson'},
    labels={
        'insurance_provider': 'Insurer',
        'billing_amount': 'Billing Amount ($)',
        'is_anomaly': 'Anomaly'
    },
    title='Billing Distribution: Normal vs Anomalous Claims per Insurer',
    height=480
)
fig3.update_layout(xaxis_tickangle=-20)
fig3.show()

In [ ]:
# ── CELL 7: Top Anomalous Claims — Audit Table ────────────────────────────────
# This is what a real claims auditor would review

audit_cols = [
    'insurance_provider', 'medical_condition', 'admission_type',
    'billing_amount', 'mean_billing', 'billing_zscore',
    'length_of_stay', 'anomaly_pct'
]

top_anomalies = (
    df_model[df_model['is_anomaly'] == 1]
    [audit_cols]
    .sort_values('anomaly_pct', ascending=False)
    .head(20)
    .rename(columns={
        'insurance_provider': 'Insurer',
        'medical_condition':  'Condition',
        'admission_type':     'Admission',
        'billing_amount':     'Billed ($)',
        'mean_billing':       'Expected ($)',
        'billing_zscore':     'Z-score',
        'length_of_stay':     'LOS (days)',
        'anomaly_pct':        'Anomaly score'
    })
)

top_anomalies['Billed ($)']    = top_anomalies['Billed ($)'].map('${:,.0f}'.format)
top_anomalies['Expected ($)']  = top_anomalies['Expected ($)'].map('${:,.0f}'.format)
top_anomalies['Z-score']       = top_anomalies['Z-score'].round(2)
top_anomalies['Anomaly score'] = top_anomalies['Anomaly score'].map('{:.1f}/100'.format)

print('TOP 20 MOST ANOMALOUS CLAIMS — FOR AUDIT REVIEW')
print('=' * 80)
print(top_anomalies.to_string(index=False))

In [ ]:
# ── CELL 8: Model Explainability — Why is a claim flagged? ───────────────────
# SHAP values show which features drove the anomaly score for each claim

try:
    import shap
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'shap', '-q'])
    import shap

# Use TreeExplainer — works natively with Isolation Forest
explainer   = shap.TreeExplainer(iso_forest)
shap_values = explainer.shap_values(X_scaled)

print('SHAP Feature Importance — What drives anomaly detection?')
print('(Higher absolute value = more influence on flagging a claim as anomalous)')

# Summary plot — shows impact of each feature across all anomalous claims
shap.summary_plot(
    shap_values[df_model['is_anomaly'] == 1],
    X[df_model['is_anomaly'] == 1],
    feature_names=FEATURES,
    plot_type='bar',
    show=True
)

# Waterfall plot for the single most anomalous claim
most_anomalous_idx = df_model['anomaly_pct'].idxmax()
print(f'\nWaterfall explanation for the most anomalous claim (index {most_anomalous_idx}):')
shap.plots._waterfall.waterfall_legacy(
    explainer.expected_value,
    shap_values[most_anomalous_idx],
    feature_names=FEATURES
)

In [ ]:
# ── CELL 9: Summary Statistics for Report & Presentation ─────────────────────
print('=' * 60)
print('FINAL SUMMARY — KEY FINDINGS')
print('=' * 60)

print(f"""
Dataset         : {n_total:,} healthcare claims
Anomalies found : {n_anomalies:,} ({n_anomalies/n_total*100:.1f}% of all claims)

Avg billing — normal claims   : ${normal['billing_amount'].mean():,.2f}
Avg billing — anomalous claims: ${anomalies['billing_amount'].mean():,.2f}
Billing uplift in anomalies   : {anomalies['billing_amount'].mean()/normal['billing_amount'].mean():.2f}x

Most flagged insurer : {anomaly_by_insurer['anomaly_count'].idxmax()}
  Anomaly rate       : {anomaly_by_insurer['anomaly_rate'].max():.1f}%
  Avg billing        : ${anomaly_by_insurer.loc[anomaly_by_insurer['anomaly_rate'].idxmax(), 'avg_billing']:,.2f}
""")

print('Anomaly breakdown by medical condition:')
cond_breakdown = (
    df_model[df_model['is_anomaly']==1]
    .groupby('medical_condition')
    .agg(count=('is_anomaly','sum'), avg_billing=('billing_amount','mean'))
    .sort_values('count', ascending=False)
    .round(2)
)
print(cond_breakdown.to_string())

print('\n✓ Analysis complete. Use charts above for your presentation.')